# CIFAR-10 Small ViT — SGD, AdamW, and Muon baselines

This notebook trains the **same small Vision Transformer from scratch on CIFAR-10** with three optimizer baselines: SGD + Nesterov momentum, AdamW, and Muon + auxiliary AdamW.

The architecture, data, augmentations, seeds, batch size, epoch budget, loss, evaluation protocol, checkpoint cadence, and WeightWatcher diagnostics are held fixed. Only optimizer-specific hyperparameters differ.

Default model: 32×32 input, 4×4 patches, embedding dimension 192, 6 transformer blocks, 3 attention heads, MLP ratio 4. Default budget: **120 epochs × 3 seeds per optimizer**.

Training uses random crop, horizontal flip, RandAugment, mixup α=0.2, label smoothing 0.1, gradient clipping at 1.0, five-epoch linear warm-up, and cosine decay. Muon is used only for eligible hidden 2-D transformer matrices; all other parameters use auxiliary AdamW.


In [ ]:
from pathlib import Path
import os
import sys
import math
import pandas as pd
import matplotlib.pyplot as plt

ROOT = None
for path in [Path.cwd(), *Path.cwd().parents]:
    candidate = path / 'baseline'
    if (candidate / 'rg_baselines').is_dir():
        ROOT = candidate
        break
    if (path / 'rg_baselines').is_dir():
        ROOT = path
        break
if ROOT is None:
    raise RuntimeError('Run this notebook from a clone of CalculatedContent/rg_optimizers.')
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rg_baselines.vit_cifar10 import (
    DEFAULT_VIT_SEEDS,
    ViTBaselineConfig,
    choose_device,
    muon_parameter_names,
    run_vit_baseline,
    SmallViT,
    summarize_final,
)

def resolve_dir(env_name, default):
    raw = os.environ.get(env_name)
    path = Path(raw).expanduser() if raw else default
    if not path.is_absolute():
        path = Path.cwd() / path
    return path.resolve()

RUN_ROOT = resolve_dir('RG_BASELINE_RUN_ROOT', ROOT / 'runs')
DATA_DIR = resolve_dir('RG_BASELINE_DATA_DIR', ROOT / 'data')
EXPERIMENT_ROOT = RUN_ROOT / 'cifar10_vit'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = choose_device()
print('device:', DEVICE)
print('experiment root:', EXPERIMENT_ROOT)


In [ ]:
CONFIG = ViTBaselineConfig(
    epochs=120,
    batch_size=128,
    warmup_epochs=5,
    patch_size=4,
    embed_dim=192,
    depth=6,
    num_heads=3,
    mlp_ratio=4.0,
    dropout=0.1,
    mixup_alpha=0.2,
    label_smoothing=0.1,
    grad_clip=1.0,
    ww_every=10,
    checkpoint_every=10,
    sgd_lr=0.10, sgd_momentum=0.9, sgd_weight_decay=5e-4,
    adamw_lr=5e-4, adamw_weight_decay=0.05,
    muon_lr=0.02, muon_momentum=0.95, muon_weight_decay=0.01,
    muon_ns_steps=5, muon_aux_lr=3e-4,
    muon_aux_beta1=0.9, muon_aux_beta2=0.95, muon_aux_weight_decay=0.01,
)
SEEDS = DEFAULT_VIT_SEEDS
model = SmallViT(CONFIG)
print(f'parameters: {sum(p.numel() for p in model.parameters()):,}')
print('Muon hidden matrices:', len(muon_parameter_names(model)))
display(pd.DataFrame([CONFIG.__dict__]))
del model


## Run the complete baseline suite

This executes **9 trainings total**: 3 optimizers × 3 independent seeds. For a smoke test only, temporarily use `ViTBaselineConfig(epochs=2, warmup_epochs=1, ww_every=2, checkpoint_every=2)` and `SEEDS=(17,)`. The committed defaults above are the intended baseline settings.


In [ ]:
all_history = []
all_spectral = []
for optimizer_name in ('sgd_momentum', 'adamw', 'muon'):
    for seed in SEEDS:
        history, spectral = run_vit_baseline(
            optimizer_name, seed, data_dir=DATA_DIR, output_dir=EXPERIMENT_ROOT,
            config=CONFIG, device=DEVICE, progress=True,
        )
        history.insert(0, 'seed', seed)
        history.insert(0, 'optimizer', optimizer_name)
        spectral.insert(0, 'seed', seed)
        spectral.insert(0, 'optimizer', optimizer_name)
        all_history.append(history)
        all_spectral.append(spectral)

performance = pd.concat(all_history, ignore_index=True)
spectral = pd.concat(all_spectral, ignore_index=True)
performance.to_csv(EXPERIMENT_ROOT / 'performance_all_runs.csv', index=False)
spectral.to_csv(EXPERIMENT_ROOT / 'weightwatcher_all_runs.csv', index=False)
final_summary = summarize_final(performance, CONFIG.epochs)
final_summary.to_csv(EXPERIMENT_ROOT / 'final_summary_95ci.csv', index=False)
display(final_summary)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for optimizer_name, group in performance.groupby('optimizer'):
    stats = group.groupby('epoch')['test_accuracy'].agg(['mean', 'std']).reset_index()
    ax.plot(stats['epoch'], 100 * stats['mean'], label=optimizer_name)
    ax.fill_between(stats['epoch'], 100*(stats['mean']-stats['std'].fillna(0)),
                    100*(stats['mean']+stats['std'].fillna(0)), alpha=0.15)
ax.set_xlabel('Epoch')
ax.set_ylabel('CIFAR-10 test accuracy (%)')
ax.set_title('Small ViT optimizer baselines')
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(EXPERIMENT_ROOT / 'test_accuracy_comparison.png', dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
for optimizer_name, group in performance.groupby('optimizer'):
    stats = group.groupby('epoch')['test_loss'].agg(['mean', 'std']).reset_index()
    ax.plot(stats['epoch'], stats['mean'], label=optimizer_name)
    ax.fill_between(stats['epoch'], stats['mean']-stats['std'].fillna(0),
                    stats['mean']+stats['std'].fillna(0), alpha=0.15)
ax.set_xlabel('Epoch')
ax.set_ylabel('CIFAR-10 test cross-entropy')
ax.set_title('Small ViT optimizer baselines')
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(EXPERIMENT_ROOT / 'test_loss_comparison.png', dpi=160)
plt.show()


## Persisted outputs

Each optimizer/seed run saves `history.csv`, `weightwatcher_by_epoch_layer.csv`, checkpoints every 10 epochs, `config.json`, and `final_state.pt`. The suite also saves aggregate performance and WeightWatcher CSVs, a final 95% Student-t confidence-interval table, and comparison plots.

WeightWatcher is called with `analyze(ERG=True, randomize=True)` so the run records the package-provided spectral diagnostics including alpha, randomized correlation-trap counts when available, and ERG gap. No proxy trap count is substituted.
